In [1]:
import sys

sys.path.append("..")

In [ ]:
from pathlib import Path

import pandas as pd
import torch

from src.models import ImageEncoder
from src.preprocessing import ImageProcessor
from src.utils.config import load_config
from src.utils.device import get_device

PROJECT_ROOT = Path.cwd().parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "base.yaml"

config = load_config(CONFIG_PATH)

device = get_device(
    config["training"]["device"]
)

print("Device:", device)

c:\Users\AYUSH PRASAD\Desktop\Multimodal Misinformation Detector\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## Loading the processed dataset

In [3]:
processed_dir = (
    PROJECT_ROOT
    / config["paths"]["processed_dir"]
)

train_path = processed_dir / "train.csv"

train_df = pd.read_csv(train_path)

print("Dataset shape:", train_df.shape)

display(
    train_df[
        ["id", "image_path", "label"]
    ].head()
)

Dataset shape: (12000, 9)


,id,image_path,label
0,4jj0fh,images/train/4jj0fh.jpg,0
1,7tykik,images/train/7tykik.jpg,0
2,1jfqnp,images/train/1jfqnp.jpg,1
3,boa5ua,images/train/boa5ua.jpg,0
4,1aco8p,images/train/1aco8p.jpg,0


In [4]:
sample_path = (
    processed_dir
    / train_df.loc[0, "image_path"]
)

print(sample_path)
print("Exists:", sample_path.exists())

c:\Users\AYUSH PRASAD\Desktop\Multimodal Misinformation Detector\data\processed\images\train\4jj0fh.jpg
Exists: True


## Initailizing Image Processor

In [5]:
processor = ImageProcessor()

print("Image processor initialized.")

Image processor initialized.


In [6]:
# Testing on a sample

sample_image = processor(sample_path)

print("Processed image shape:", tuple(sample_image.shape))
print("Dtype:", sample_image.dtype)

Processed image shape: (3, 224, 224)
Dtype: torch.float32


In [7]:
# Testing the encoder

encoder = ImageEncoder(
    freeze=config["models"]["image"]["freeze"]
)

encoder = encoder.to(device)

print(
    "Embedding dimension:",
    encoder.embedding_dim,
)

Embedding dimension: 512


## Testing on a small batch

In [8]:
sample_paths = [
    processed_dir / path
    for path in train_df["image_path"].head(4)
]

batch = torch.stack(
    [
        processor(path)
        for path in sample_paths
    ]
)

print("Batch shape:", tuple(batch.shape))

Batch shape: (4, 3, 224, 224)


## Moving batch to GPU

In [9]:
batch = batch.to(device)

print("Batch device:", batch.device)

Batch device: cuda:0


## Generate image embeddings

In [10]:
with torch.inference_mode():
    embeddings = encoder(batch)

print(
    "Embedding shape:",
    tuple(embeddings.shape),
)

print(
    "Embedding dtype:",
    embeddings.dtype,
)

print(
    "Device:",
    embeddings.device,
)

Embedding shape: (4, 512)
Embedding dtype: torch.float32
Device: cuda:0


## Verifying that ResNet is frozen

In [11]:
trainable_parameters = sum(
    parameter.numel()
    for parameter in encoder.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in encoder.parameters()
)

print(
    "Total parameters:    ",
    f"{total_parameters:,}",
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}",
)

Total parameters:     11,176,512
Trainable parameters: 0


## Final sanity checks

In [12]:
assert embeddings.ndim == 2
assert embeddings.shape[0] == len(sample_paths)
assert embeddings.shape[1] == encoder.embedding_dim
assert embeddings.shape[1] == 512
assert trainable_parameters == 0
assert all(path.exists() for path in sample_paths)

print("Image pipeline sanity checks: PASSED")

Image pipeline sanity checks: PASSED


Phase 2 provided us the text pipeline and Phase 3 has provided us the image pipeline.

So, now we have:

```
                 INPUT
                /     \
               /       \
           TEXT         IMAGE
            │             │
       DistilBERT      ResNet-18
            │             │
          768-D         512-D
         embedding     embedding
```


In phase 4 we will pass 768 + 512 = 1280 dimensions
to our fusion model